## 1. Imports

In [2]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath(".."))

# EarlyFusion models
from EarlyFusion.EarlyFusionMLP import EarlyFusionMLP
from EarlyFusion.EarlyFusionCNN import EarlyFusionCNN
from EarlyFusion.utils import cross_validation_5fold_early_fusion

## 2. Data Loading

### 2.1 Load Image Data

In [ ]:
# [TODO] Define paths to image data and labels
images_dir = "path/to/images_directory"
labels_csv = "path/to/labels.csv"
train_folds_csv = "path/to/train_folds.csv"
val_test_folds_csv = "path/to/val_test_folds.csv"

# [TODO] Load images (example: using PIL or cv2)
# For demonstration, assuming you have a function to load images
# images = load_images_from_directory(images_dir)

# Load metadata
labels = pd.read_csv(labels_csv)
train_folds = pd.read_csv(train_folds_csv)
val_test_folds = pd.read_csv(val_test_folds_csv)

print(f"Labels shape: {labels.shape}")
print(f"Train folds shape: {train_folds.shape}")
print(f"Val/Test folds shape: {val_test_folds.shape}")

### 2.2 Load Extracted Features Data

In [ ]:
# [TODO] Define paths to extracted features
extracted_features_csv = "path/to/extracted_features.csv"

# Load extracted features
extracted_features = pd.read_csv(extracted_features_csv).values

print(f"Extracted features shape: {extracted_features.shape}")
print(f"Features per sample: {extracted_features.shape[1]}")

## 3. Concatenation

### 3.1 Image Data Processing

In [ ]:
# [TODO] Process images for concatenation
# This depends on your image format and preprocessing pipeline
# Example: flatten images or convert to feature vectors

# For now, assuming images are already loaded and processed
# images_processed = preprocess_images(images)  # Adjust based on your needs
# images_flattened = images_processed.reshape(images_processed.shape[0], -1)

print("[TODO] Process and flatten image data")
# print(f"Processed images shape: {images_flattened.shape}")

### 3.2 Feature Concatenation

In [ ]:
# Save extracted features as CSV for CNN evaluation
extracted_features_csv_path = "path/to/concatenated_extracted_features.csv"
extracted_features_df = pd.DataFrame(extracted_features)
# extracted_features_df.to_csv(extracted_features_csv_path, index=False)

print(f"Extracted features saved to: {extracted_features_csv_path}")
print(f"Extracted features shape: {extracted_features_df.shape}")

## 4. Cross-Validation

### 4.1 CNN Cross-Validation on Image Data

In [ ]:
# [TODO] Define output directory for CNN models
output_dir_cnn = "path/to/models/cnn_images"

# [TODO] Prepare image data as CSV if needed for the CV function
images_csv = "path/to/images_features.csv"
# images_df = pd.DataFrame(images_flattened)
# images_df.to_csv(images_csv, index=False)

print("Running 5-fold CV with EarlyFusionCNN on Image Data...")
best_model_path_cnn_images, metrics_history_cnn_images = cross_validation_5fold_early_fusion(
    features_csv=images_csv,
    labels_csv=labels_csv,
    train_folds_csv=train_folds_csv,
    val_test_folds_csv=val_test_folds_csv,
    output_dir=output_dir_cnn,
    NN_type="CNN",
    hidden_dims=[512, 256],
    batch_size=64,
    num_epochs=100,
    learning_rate=0.001,
    weight_decay=0.0,
    dropout=0.5,
    pooling_type="max",
    verbose=True
)
print(f"Best CNN model (images) saved at: {best_model_path_cnn_images}")

### 4.2 MLP Cross-Validation on Extracted Features

In [ ]:
# [TODO] Define output directory for MLP models
output_dir_mlp = "path/to/models/mlp_features"

print("Running 5-fold CV with EarlyFusionMLP on Extracted Features...")
best_model_path_mlp_features, metrics_history_mlp_features = cross_validation_5fold_early_fusion(
    features_csv=extracted_features_csv_path,
    labels_csv=labels_csv,
    train_folds_csv=train_folds_csv,
    val_test_folds_csv=val_test_folds_csv,
    output_dir=output_dir_mlp,
    NN_type="MLP",
    hidden_dims=[512, 256, 128],
    batch_size=64,
    num_epochs=100,
    learning_rate=0.001,
    weight_decay=0.0,
    dropout=0.5,
    verbose=True
)
print(f"Best MLP model (features) saved at: {best_model_path_mlp_features}")

## 5. Results Visualization

### 5.1 Image Data Results

In [ ]:
# Set up matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Prepare data for CNN (images) visualization
cnn_folds = metrics_history_cnn_images['folds']
cnn_acc = metrics_history_cnn_images['test_acc']
cnn_f1 = metrics_history_cnn_images['test_f1']
cnn_auc = metrics_history_cnn_images['test_roc_auc']
cnn_loss = metrics_history_cnn_images['test_loss']

print("CNN Results (Image Data):")
print(f"  Accuracy: {np.mean(cnn_acc):.4f} ± {np.std(cnn_acc):.4f}")
print(f"  F1 Score: {np.mean(cnn_f1):.4f} ± {np.std(cnn_f1):.4f}")
print(f"  ROC AUC:  {np.mean(cnn_auc):.4f} ± {np.std(cnn_auc):.4f}")
print(f"  Loss:     {np.mean(cnn_loss):.4f} ± {np.std(cnn_loss):.4f}")

In [ ]:
# Create CNN (images) comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cross-Validation Results: CNN on Image Data', fontsize=16, fontweight='bold')

# Plot 1: Accuracy
axes[0, 0].plot(cnn_folds, cnn_acc, marker='o', linewidth=2, markersize=8, color='steelblue')
axes[0, 0].axhline(np.mean(cnn_acc), color='steelblue', linestyle='--', alpha=0.5)
axes[0, 0].fill_between(cnn_folds, np.mean(cnn_acc) - np.std(cnn_acc), 
                         np.mean(cnn_acc) + np.std(cnn_acc), alpha=0.2, color='steelblue')
axes[0, 0].set_xlabel('Fold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Test Accuracy per Fold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: F1 Score
axes[0, 1].plot(cnn_folds, cnn_f1, marker='o', linewidth=2, markersize=8, color='darkgreen')
axes[0, 1].axhline(np.mean(cnn_f1), color='darkgreen', linestyle='--', alpha=0.5)
axes[0, 1].fill_between(cnn_folds, np.mean(cnn_f1) - np.std(cnn_f1), 
                         np.mean(cnn_f1) + np.std(cnn_f1), alpha=0.2, color='darkgreen')
axes[0, 1].set_xlabel('Fold')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('Test F1 Score per Fold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: ROC AUC
axes[1, 0].plot(cnn_folds, cnn_auc, marker='o', linewidth=2, markersize=8, color='darkorange')
axes[1, 0].axhline(np.mean(cnn_auc), color='darkorange', linestyle='--', alpha=0.5)
axes[1, 0].fill_between(cnn_folds, np.mean(cnn_auc) - np.std(cnn_auc), 
                         np.mean(cnn_auc) + np.std(cnn_auc), alpha=0.2, color='darkorange')
axes[1, 0].set_xlabel('Fold')
axes[1, 0].set_ylabel('ROC AUC')
axes[1, 0].set_title('Test ROC AUC per Fold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Loss
axes[1, 1].plot(cnn_folds, cnn_loss, marker='o', linewidth=2, markersize=8, color='crimson')
axes[1, 1].axhline(np.mean(cnn_loss), color='crimson', linestyle='--', alpha=0.5)
axes[1, 1].fill_between(cnn_folds, np.mean(cnn_loss) - np.std(cnn_loss), 
                         np.mean(cnn_loss) + np.std(cnn_loss), alpha=0.2, color='crimson')
axes[1, 1].set_xlabel('Fold')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Test Loss per Fold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Extracted Features Results

In [ ]:
# Prepare data for MLP (features) visualization
mlp_folds = metrics_history_mlp_features['folds']
mlp_acc = metrics_history_mlp_features['test_acc']
mlp_f1 = metrics_history_mlp_features['test_f1']
mlp_auc = metrics_history_mlp_features['test_roc_auc']
mlp_loss = metrics_history_mlp_features['test_loss']

print("MLP Results (Extracted Features):")
print(f"  Accuracy: {np.mean(mlp_acc):.4f} ± {np.std(mlp_acc):.4f}")
print(f"  F1 Score: {np.mean(mlp_f1):.4f} ± {np.std(mlp_f1):.4f}")
print(f"  ROC AUC:  {np.mean(mlp_auc):.4f} ± {np.std(mlp_auc):.4f}")
print(f"  Loss:     {np.mean(mlp_loss):.4f} ± {np.std(mlp_loss):.4f}")

In [ ]:
# Create MLP (features) comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cross-Validation Results: MLP on Extracted Features', fontsize=16, fontweight='bold')

# Plot 1: Accuracy
axes[0, 0].plot(mlp_folds, mlp_acc, marker='s', linewidth=2, markersize=8, color='steelblue')
axes[0, 0].axhline(np.mean(mlp_acc), color='steelblue', linestyle='--', alpha=0.5)
axes[0, 0].fill_between(mlp_folds, np.mean(mlp_acc) - np.std(mlp_acc), 
                         np.mean(mlp_acc) + np.std(mlp_acc), alpha=0.2, color='steelblue')
axes[0, 0].set_xlabel('Fold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Test Accuracy per Fold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: F1 Score
axes[0, 1].plot(mlp_folds, mlp_f1, marker='s', linewidth=2, markersize=8, color='darkgreen')
axes[0, 1].axhline(np.mean(mlp_f1), color='darkgreen', linestyle='--', alpha=0.5)
axes[0, 1].fill_between(mlp_folds, np.mean(mlp_f1) - np.std(mlp_f1), 
                         np.mean(mlp_f1) + np.std(mlp_f1), alpha=0.2, color='darkgreen')
axes[0, 1].set_xlabel('Fold')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('Test F1 Score per Fold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: ROC AUC
axes[1, 0].plot(mlp_folds, mlp_auc, marker='s', linewidth=2, markersize=8, color='darkorange')
axes[1, 0].axhline(np.mean(mlp_auc), color='darkorange', linestyle='--', alpha=0.5)
axes[1, 0].fill_between(mlp_folds, np.mean(mlp_auc) - np.std(mlp_auc), 
                         np.mean(mlp_auc) + np.std(mlp_auc), alpha=0.2, color='darkorange')
axes[1, 0].set_xlabel('Fold')
axes[1, 0].set_ylabel('ROC AUC')
axes[1, 0].set_title('Test ROC AUC per Fold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Loss
axes[1, 1].plot(mlp_folds, mlp_loss, marker='s', linewidth=2, markersize=8, color='crimson')
axes[1, 1].axhline(np.mean(mlp_loss), color='crimson', linestyle='--', alpha=0.5)
axes[1, 1].fill_between(mlp_folds, np.mean(mlp_loss) - np.std(mlp_loss), 
                         np.mean(mlp_loss) + np.std(mlp_loss), alpha=0.2, color='crimson')
axes[1, 1].set_xlabel('Fold')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Test Loss per Fold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.3 Comparison: Images vs Features

In [ ]:
# Create comparison summary statistics plot
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ['Accuracy', 'F1 Score', 'ROC AUC', 'Loss']
cnn_means = [np.mean(cnn_acc), np.mean(cnn_f1), np.mean(cnn_auc), np.mean(cnn_loss)]
cnn_stds = [np.std(cnn_acc), np.std(cnn_f1), np.std(cnn_auc), np.std(cnn_loss)]
mlp_means = [np.mean(mlp_acc), np.mean(mlp_f1), np.mean(mlp_auc), np.mean(mlp_loss)]
mlp_stds = [np.std(mlp_acc), np.std(mlp_f1), np.std(mlp_auc), np.std(mlp_loss)]

x = np.arange(len(metrics))
width = 0.35

ax.bar(x - width/2, cnn_means, width, label='CNN (Images)', yerr=cnn_stds, capsize=5, alpha=0.8, color='steelblue')
ax.bar(x + width/2, mlp_means, width, label='MLP (Features)', yerr=mlp_stds, capsize=5, alpha=0.8, color='darkgreen')

ax.set_ylabel('Metric Value', fontsize=12)
ax.set_title('Average Cross-Validation Performance: CNN (Images) vs MLP (Features)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()